In [0]:
# ============================================================
# 🚗 AUTOMOBILE SHOWROOM ANALYTICS
# 🥇 GOLD LAYER TRANSFORMATION
# ============================================================
#
# Purpose:
# Create business-ready analytical datasets from Silver tables
# for Azure Synapse and Power BI.
#
# Architecture:
# Bronze → Silver → Gold → Synapse → Power BI
#
# Gold objectives:
# 1. Sales Performance
# 2. Showroom Profitability
# 3. Vehicle & Inventory Performance
# 4. Salesperson Performance
# 5. Marketing & Lead Performance
# ============================================================

In [0]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")

Libraries imported successfully.


In [0]:
# ============================================================
# SET PROJECT CATALOG
# ============================================================

spark.sql(
    "USE CATALOG showroom_analytics"
)

print("Catalog selected: showroom_analytics")

Catalog selected: showroom_analytics


In [0]:
# ============================================================
# CREATE GOLD SCHEMA
# ============================================================

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS showroom_analytics.gold
""")

print("Gold schema is ready.")

Gold schema is ready.


In [0]:
# ============================================================
# LOAD SILVER TABLES
# ============================================================

df_sales = spark.table(
    "showroom_analytics.silver.sales"
)

df_inventory = spark.table(
    "showroom_analytics.silver.inventory"
)

df_expenses = spark.table(
    "showroom_analytics.silver.expenses"
)

df_campaign_expense = spark.table(
    "showroom_analytics.silver.campaign_expense"
)

df_leads = spark.table(
    "showroom_analytics.silver.marketing_lead"
)

df_followup = spark.table(
    "showroom_analytics.silver.customer_followup"
)

df_test_drive = spark.table(
    "showroom_analytics.silver.test_drive"
)

df_vehicle = spark.table(
    "showroom_analytics.silver.vehicle"
)

df_vehicle_model = spark.table(
    "showroom_analytics.silver.vehicle_model"
)

df_showroom = spark.table(
    "showroom_analytics.silver.showroom"
)

df_salesperson = spark.table(
    "showroom_analytics.silver.salesperson"
)

df_customer = spark.table(
    "showroom_analytics.silver.customer"
)

df_campaign = spark.table(
    "showroom_analytics.silver.campaign"
)

print("✅ All Silver tables loaded successfully.")

✅ All Silver tables loaded successfully.


In [0]:
# ============================================================
# ERIFY SILVER TABLES
# ============================================================

silver_tables = [
    "sales",
    "inventory",
    "expenses",
    "campaign_expense",
    "marketing_lead",
    "customer_followup",
    "test_drive",
    "vehicle",
    "vehicle_model",
    "showroom",
    "salesperson",
    "customer",
    "campaign"
]

for table_name in silver_tables:
    count = spark.table(
        f"showroom_analytics.silver.{table_name}"
    ).count()

    print(f" {table_name}: {count} records")

 sales: 11 records
 inventory: 20 records
 expenses: 15 records
 campaign_expense: 15 records
 marketing_lead: 20 records
 customer_followup: 20 records
 test_drive: 20 records
 vehicle: 38 records
 vehicle_model: 28 records
 showroom: 15 records
 salesperson: 15 records
 customer: 20 records
 campaign: 10 records


In [0]:
# ============================================================
# FACT SALES - BUILD GOLD TABLE
# ============================================================
#
# Purpose:
# Combine sales transactions with vehicle, model, showroom,
# customer and salesperson information.
#
# This table will support:
# - Revenue analysis
# - Gross profit
# - Gross margin
# - Vehicle performance
# - Showroom performance
# - Salesperson performance
# - Monthly sales trends
# ============================================================

df_fact_sales = (
    df_sales.alias("s")

    # Vehicle
    .join(
        df_vehicle.alias("v"),
        F.col("s.vehicle_id") == F.col("v.vehicle_id"),
        "left"
    )

    # Vehicle model
    .join(
        df_vehicle_model.alias("vm"),
        F.col("v.model_id") == F.col("vm.model_id"),
        "left"
    )

    # Showroom
    .join(
        df_showroom.alias("sh"),
        F.col("s.showroom_id") == F.col("sh.showroom_id"),
        "left"
    )

    # Customer
    .join(
        df_customer.alias("c"),
        F.col("s.customer_id") == F.col("c.customer_id"),
        "left"
    )

    # Salesperson
    .join(
        df_salesperson.alias("sp"),
        F.col("s.salesperson_id") == F.col("sp.salesperson_id"),
        "left"
    )

    .select(
        # Sale
        F.col("s.sale_id"),
        F.col("s.sale_date"),

        # Customer
        F.col("s.customer_id"),
        F.col("c.customer_name"),

        # Vehicle
        F.col("s.vehicle_id"),
        F.col("v.variant"),
        F.col("v.fuel_type"),
        F.col("v.transmission"),
        F.col("v.manufacturing_year"),

        # Vehicle model
        F.col("v.model_id"),
        F.col("vm.brand"),
        F.col("vm.model_name"),
        F.col("vm.vehicle_type"),
        F.col("vm.segment"),

        # Showroom
        F.col("s.showroom_id"),
        F.col("sh.showroom_name"),
        F.col("sh.city"),
        F.col("sh.state"),

        # Salesperson
        F.col("s.salesperson_id"),
        F.col("sp.salesperson_name"),

        # Financial fields
        F.col("s.sale_price"),
        F.col("s.purchase_cost"),
        F.col("s.discount"),
        F.col("s.commission"),
        F.col("s.payment_method")
    )
)

display(df_fact_sales)

sale_id,sale_date,customer_id,customer_name,vehicle_id,variant,fuel_type,transmission,manufacturing_year,model_id,brand,model_name,vehicle_type,segment,showroom_id,showroom_name,city,state,salesperson_id,salesperson_name,sale_price,purchase_cost,discount,commission,payment_method
1,2026-01-10,1,Sachin Kaware,10001,XZ+,Petrol,Manual,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,1,Rahul Patil,1180000.00,1050000.00,20000.00,15000.00,Loan
2,2026-01-20,2,Amit Sharma,10002,XZ+ Lux,Petrol,Automatic,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,2,Amit Sharma,1350000.00,1200000.00,30000.00,18000.00,Bank Transfer
3,2026-01-25,3,Priya Deshmukh,10008,S,Petrol,Manual,2026,105,Hyundai,Creta,Suv,Mid,2,Mumbai Andheri,Mumbai,MAHARASHTRA,3,Priya Deshmukh,1680000.00,1500000.00,25000.00,20000.00,Loan
4,2026-02-12,5,Sneha Kulkarni,10016,VXI,Petrol,Manual,2026,109,Maruti,Swift,Hatchback,Entry,3,Nashik Road,Nashik,MAHARASHTRA,5,Sneha Kulkarni,830000.00,720000.00,10000.00,9000.00,Upi
5,2026-02-22,7,Neha Patil,10023,AX5,Petrol,Manual,2026,113,Mahindra,XUV 3XO,Suv,Mid,4,Nagpur Central,Nagpur,MAHARASHTRA,7,Sagar More,1420000.00,1200000.00,30000.00,16000.00,Loan
6,2026-03-02,8,Kunal Shinde,10027,HTX,Petrol,Automatic,2026,117,Kia,Seltos,Suv,Mid,5,Thane West,Thane,MAHARASHTRA,9,Kunal Shinde,1760000.00,1600000.00,20000.00,18000.00,Bank Transfer
7,2026-03-12,10,Sagar More,10030,ZX,Hybrid,Automatic,2026,120,Toyota,Innova,Muv,Premium,6,Navi Mumbai,Navi Mumbai,MAHARASHTRA,11,Akash Gaikwad,2450000.00,2200000.00,50000.00,25000.00,Loan
8,2026-03-25,13,Vikas Jadhav,10034,ZX,Petrol,Automatic,2026,124,Honda,Elevate,Suv,Mid,7,Aurangabad City,Aurangabad,MAHARASHTRA,13,Nitin Kadam,1690000.00,1550000.00,25000.00,17000.00,Upi
9,2026-04-05,15,Manish Yadav,10037,GT,Petrol,Automatic,2026,127,Volkswagen,Taigun,Suv,Mid,9,Ahmednagar Road,Ahmednagar,MAHARASHTRA,15,Manish Yadav,1880000.00,1700000.00,30000.00,19000.00,Loan
10,2026-04-18,18,Kiran Patil,10005,Creative+,Petrol,Automatic,2026,102,Tata,Punch,Suv,Entry,3,Nashik Road,Nashik,MAHARASHTRA,6,Rohit Joshi,1080000.00,950000.00,15000.00,10000.00,Cash


In [0]:
# ============================================================
# FACT SALES - BUSINESS METRICS
# ============================================================

df_fact_sales = (
    df_fact_sales

    # Revenue
    .withColumn(
        "revenue",
        F.col("sale_price")
    )

    # Gross profit
    .withColumn(
        "gross_profit",
        F.col("sale_price")
        - F.col("purchase_cost")
        - F.col("discount")
        - F.col("commission")
    )

    # Gross margin percentage
    .withColumn(
        "gross_margin_percentage",
        F.when(
            F.col("sale_price") > 0,
            (
                F.col("gross_profit")
                / F.col("sale_price")
            ) * 100
        ).otherwise(0)
    )

    # Date attributes
    .withColumn(
        "sale_year",
        F.year("sale_date")
    )
    .withColumn(
        "sale_month",
        F.month("sale_date")
    )
    .withColumn(
        "sale_month_name",
        F.date_format("sale_date", "MMMM")
    )
    .withColumn(
        "sale_quarter",
        F.quarter("sale_date")
    )
)

display(df_fact_sales)

sale_id,sale_date,customer_id,customer_name,vehicle_id,variant,fuel_type,transmission,manufacturing_year,model_id,brand,model_name,vehicle_type,segment,showroom_id,showroom_name,city,state,salesperson_id,salesperson_name,sale_price,purchase_cost,discount,commission,payment_method,revenue,gross_profit,gross_margin_percentage,sale_year,sale_month,sale_month_name,sale_quarter
1,2026-01-10,1,Sachin Kaware,10001,XZ+,Petrol,Manual,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,1,Rahul Patil,1180000.00,1050000.00,20000.00,15000.00,Loan,1180000.00,95000.00,8.0508474576271,2026,1,January,1
2,2026-01-20,2,Amit Sharma,10002,XZ+ Lux,Petrol,Automatic,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,2,Amit Sharma,1350000.00,1200000.00,30000.00,18000.00,Bank Transfer,1350000.00,102000.00,7.5555555555556,2026,1,January,1
3,2026-01-25,3,Priya Deshmukh,10008,S,Petrol,Manual,2026,105,Hyundai,Creta,Suv,Mid,2,Mumbai Andheri,Mumbai,MAHARASHTRA,3,Priya Deshmukh,1680000.00,1500000.00,25000.00,20000.00,Loan,1680000.00,135000.00,8.0357142857143,2026,1,January,1
4,2026-02-12,5,Sneha Kulkarni,10016,VXI,Petrol,Manual,2026,109,Maruti,Swift,Hatchback,Entry,3,Nashik Road,Nashik,MAHARASHTRA,5,Sneha Kulkarni,830000.00,720000.00,10000.00,9000.00,Upi,830000.00,91000.00,10.9638554216867,2026,2,February,1
5,2026-02-22,7,Neha Patil,10023,AX5,Petrol,Manual,2026,113,Mahindra,XUV 3XO,Suv,Mid,4,Nagpur Central,Nagpur,MAHARASHTRA,7,Sagar More,1420000.00,1200000.00,30000.00,16000.00,Loan,1420000.00,174000.00,12.2535211267606,2026,2,February,1
6,2026-03-02,8,Kunal Shinde,10027,HTX,Petrol,Automatic,2026,117,Kia,Seltos,Suv,Mid,5,Thane West,Thane,MAHARASHTRA,9,Kunal Shinde,1760000.00,1600000.00,20000.00,18000.00,Bank Transfer,1760000.00,122000.00,6.9318181818182,2026,3,March,1
7,2026-03-12,10,Sagar More,10030,ZX,Hybrid,Automatic,2026,120,Toyota,Innova,Muv,Premium,6,Navi Mumbai,Navi Mumbai,MAHARASHTRA,11,Akash Gaikwad,2450000.00,2200000.00,50000.00,25000.00,Loan,2450000.00,175000.00,7.1428571428571,2026,3,March,1
8,2026-03-25,13,Vikas Jadhav,10034,ZX,Petrol,Automatic,2026,124,Honda,Elevate,Suv,Mid,7,Aurangabad City,Aurangabad,MAHARASHTRA,13,Nitin Kadam,1690000.00,1550000.00,25000.00,17000.00,Upi,1690000.00,98000.00,5.7988165680473,2026,3,March,1
9,2026-04-05,15,Manish Yadav,10037,GT,Petrol,Automatic,2026,127,Volkswagen,Taigun,Suv,Mid,9,Ahmednagar Road,Ahmednagar,MAHARASHTRA,15,Manish Yadav,1880000.00,1700000.00,30000.00,19000.00,Loan,1880000.00,131000.00,6.9680851063830,2026,4,April,2
10,2026-04-18,18,Kiran Patil,10005,Creative+,Petrol,Automatic,2026,102,Tata,Punch,Suv,Entry,3,Nashik Road,Nashik,MAHARASHTRA,6,Rohit Joshi,1080000.00,950000.00,15000.00,10000.00,Cash,1080000.00,105000.00,9.7222222222222,2026,4,April,2


In [0]:
# ============================================================
# FACT SALES - WRITE TO GOLD
# ============================================================

df_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.fact_sales"
    )

print(
    "SUCCESS: fact_sales created in Gold."
)

SUCCESS: fact_sales created in Gold.


In [0]:
# ============================================================
# FACT SALES - VERIFY GOLD TABLE
# ============================================================

df_gold_fact_sales = spark.table(
    "showroom_analytics.gold.fact_sales"
)

display(df_gold_fact_sales)

print(
    "Gold fact_sales records:",
    df_gold_fact_sales.count()
)

df_gold_fact_sales.printSchema()

sale_id,sale_date,customer_id,customer_name,vehicle_id,variant,fuel_type,transmission,manufacturing_year,model_id,brand,model_name,vehicle_type,segment,showroom_id,showroom_name,city,state,salesperson_id,salesperson_name,sale_price,purchase_cost,discount,commission,payment_method,revenue,gross_profit,gross_margin_percentage,sale_year,sale_month,sale_month_name,sale_quarter
1,2026-01-10,1,Sachin Kaware,10001,XZ+,Petrol,Manual,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,1,Rahul Patil,1180000.00,1050000.00,20000.00,15000.00,Loan,1180000.00,95000.00,8.0508474576271,2026,1,January,1
2,2026-01-20,2,Amit Sharma,10002,XZ+ Lux,Petrol,Automatic,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,2,Amit Sharma,1350000.00,1200000.00,30000.00,18000.00,Bank Transfer,1350000.00,102000.00,7.5555555555556,2026,1,January,1
3,2026-01-25,3,Priya Deshmukh,10008,S,Petrol,Manual,2026,105,Hyundai,Creta,Suv,Mid,2,Mumbai Andheri,Mumbai,MAHARASHTRA,3,Priya Deshmukh,1680000.00,1500000.00,25000.00,20000.00,Loan,1680000.00,135000.00,8.0357142857143,2026,1,January,1
4,2026-02-12,5,Sneha Kulkarni,10016,VXI,Petrol,Manual,2026,109,Maruti,Swift,Hatchback,Entry,3,Nashik Road,Nashik,MAHARASHTRA,5,Sneha Kulkarni,830000.00,720000.00,10000.00,9000.00,Upi,830000.00,91000.00,10.9638554216867,2026,2,February,1
5,2026-02-22,7,Neha Patil,10023,AX5,Petrol,Manual,2026,113,Mahindra,XUV 3XO,Suv,Mid,4,Nagpur Central,Nagpur,MAHARASHTRA,7,Sagar More,1420000.00,1200000.00,30000.00,16000.00,Loan,1420000.00,174000.00,12.2535211267606,2026,2,February,1
6,2026-03-02,8,Kunal Shinde,10027,HTX,Petrol,Automatic,2026,117,Kia,Seltos,Suv,Mid,5,Thane West,Thane,MAHARASHTRA,9,Kunal Shinde,1760000.00,1600000.00,20000.00,18000.00,Bank Transfer,1760000.00,122000.00,6.9318181818182,2026,3,March,1
7,2026-03-12,10,Sagar More,10030,ZX,Hybrid,Automatic,2026,120,Toyota,Innova,Muv,Premium,6,Navi Mumbai,Navi Mumbai,MAHARASHTRA,11,Akash Gaikwad,2450000.00,2200000.00,50000.00,25000.00,Loan,2450000.00,175000.00,7.1428571428571,2026,3,March,1
8,2026-03-25,13,Vikas Jadhav,10034,ZX,Petrol,Automatic,2026,124,Honda,Elevate,Suv,Mid,7,Aurangabad City,Aurangabad,MAHARASHTRA,13,Nitin Kadam,1690000.00,1550000.00,25000.00,17000.00,Upi,1690000.00,98000.00,5.7988165680473,2026,3,March,1
9,2026-04-05,15,Manish Yadav,10037,GT,Petrol,Automatic,2026,127,Volkswagen,Taigun,Suv,Mid,9,Ahmednagar Road,Ahmednagar,MAHARASHTRA,15,Manish Yadav,1880000.00,1700000.00,30000.00,19000.00,Loan,1880000.00,131000.00,6.9680851063830,2026,4,April,2
10,2026-04-18,18,Kiran Patil,10005,Creative+,Petrol,Automatic,2026,102,Tata,Punch,Suv,Entry,3,Nashik Road,Nashik,MAHARASHTRA,6,Rohit Joshi,1080000.00,950000.00,15000.00,10000.00,Cash,1080000.00,105000.00,9.7222222222222,2026,4,April,2


Gold fact_sales records: 11
root
 |-- sale_id: long (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- variant: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- manufacturing_year: integer (nullable = true)
 |-- model_id: integer (nullable = true)
 |-- brand: string (nullable = true)
 |-- model_name: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- showroom_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- salesperson_name: string (nullable = true)
 |-- sale_price: decimal(18,2) (nullable = true)
 |-- purchase_cost: decimal(18,2) (nullable = true)
 |-- discount: decimal(18,

In [0]:
# ============================================================
# FACT INVENTORY - BUILD GOLD TABLE
# ============================================================

df_fact_inventory = (
    df_inventory.alias("i")

    # Vehicle
    .join(
        df_vehicle.alias("v"),
        F.col("i.vehicle_id") == F.col("v.vehicle_id"),
        "left"
    )

    # Vehicle model
    .join(
        df_vehicle_model.alias("vm"),
        F.col("v.model_id") == F.col("vm.model_id"),
        "left"
    )

    # Showroom
    .join(
        df_showroom.alias("sh"),
        F.col("i.showroom_id") == F.col("sh.showroom_id"),
        "left"
    )

    .select(
        # Inventory
        F.col("i.inventory_id"),
        F.col("i.purchase_date"),
        F.col("i.status"),
        F.col("i.vin"),

        # Vehicle
        F.col("i.vehicle_id"),
        F.col("v.variant"),
        F.col("v.fuel_type"),
        F.col("v.transmission"),
        F.col("v.manufacturing_year"),

        # Model
        F.col("v.model_id"),
        F.col("vm.brand"),
        F.col("vm.model_name"),
        F.col("vm.vehicle_type"),
        F.col("vm.segment"),

        # Showroom
        F.col("i.showroom_id"),
        F.col("sh.showroom_name"),
        F.col("sh.city"),
        F.col("sh.state"),

        # Financial
        F.col("i.purchase_cost")
    )
)

display(df_fact_inventory)

inventory_id,purchase_date,status,vin,vehicle_id,variant,fuel_type,transmission,manufacturing_year,model_id,brand,model_name,vehicle_type,segment,showroom_id,showroom_name,city,state,purchase_cost
1,2026-01-05,Sold,VIN20260001,10001,XZ+,Petrol,Manual,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,1050000.00
2,2026-01-12,Sold,VIN20260002,10002,XZ+ Lux,Petrol,Automatic,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,1200000.00
3,2026-01-18,Sold,VIN20260003,10008,S,Petrol,Manual,2026,105,Hyundai,Creta,Suv,Mid,2,Mumbai Andheri,Mumbai,MAHARASHTRA,1500000.00
4,2026-02-01,Available,VIN20260004,10009,SX,Petrol,Automatic,2026,105,Hyundai,Creta,Suv,Mid,2,Mumbai Andheri,Mumbai,MAHARASHTRA,1600000.00
5,2026-02-05,Sold,VIN20260005,10016,VXI,Petrol,Manual,2026,109,Maruti,Swift,Hatchback,Entry,3,Nashik Road,Nashik,MAHARASHTRA,720000.00
6,2026-02-10,Available,VIN20260006,10017,ZXI+,Petrol,Automatic,2026,109,Maruti,Swift,Hatchback,Entry,3,Nashik Road,Nashik,MAHARASHTRA,850000.00
7,2026-02-15,Sold,VIN20260007,10023,AX5,Petrol,Manual,2026,113,Mahindra,XUV 3XO,Suv,Mid,4,Nagpur Central,Nagpur,MAHARASHTRA,1200000.00
8,2026-02-20,Reserved,VIN20260008,10024,S11,Diesel,Manual,2026,114,Mahindra,Scorpio,Suv,Premium,4,Nagpur Central,Nagpur,MAHARASHTRA,1650000.00
9,2026-02-25,Sold,VIN20260009,10027,HTX,Petrol,Automatic,2026,117,Kia,Seltos,Suv,Mid,5,Thane West,Thane,MAHARASHTRA,1600000.00
10,2026-03-01,Available,VIN20260010,10028,GTX+,Petrol,Automatic,2026,118,Kia,Sonet,Suv,Entry,5,Thane West,Thane,MAHARASHTRA,1250000.00


In [0]:
# ============================================================
# FACT INVENTORY - BUSINESS METRICS
# ============================================================

df_fact_inventory = (
    df_fact_inventory

    # Inventory value
    .withColumn(
        "inventory_value",
        F.col("purchase_cost")
    )

    # Number of days vehicle has been in inventory
    .withColumn(
        "inventory_days",
        F.datediff(
            F.current_date(),
            F.col("purchase_date")
        )
    )

    # Inventory aging category
    .withColumn(
        "inventory_age_category",
        F.when(
            F.col("inventory_days") <= 30,
            "0-30 Days"
        )
        .when(
            F.col("inventory_days") <= 60,
            "31-60 Days"
        )
        .when(
            F.col("inventory_days") <= 90,
            "61-90 Days"
        )
        .otherwise(
            "90+ Days"
        )
    )
)

display(df_fact_inventory)

inventory_id,purchase_date,status,vin,vehicle_id,variant,fuel_type,transmission,manufacturing_year,model_id,brand,model_name,vehicle_type,segment,showroom_id,showroom_name,city,state,purchase_cost,inventory_value,inventory_days,inventory_age_category
1,2026-01-05,Sold,VIN20260001,10001,XZ+,Petrol,Manual,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,1050000.00,1050000.00,224,90+ Days
2,2026-01-12,Sold,VIN20260002,10002,XZ+ Lux,Petrol,Automatic,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,1200000.00,1200000.00,217,90+ Days
3,2026-01-18,Sold,VIN20260003,10008,S,Petrol,Manual,2026,105,Hyundai,Creta,Suv,Mid,2,Mumbai Andheri,Mumbai,MAHARASHTRA,1500000.00,1500000.00,211,90+ Days
4,2026-02-01,Available,VIN20260004,10009,SX,Petrol,Automatic,2026,105,Hyundai,Creta,Suv,Mid,2,Mumbai Andheri,Mumbai,MAHARASHTRA,1600000.00,1600000.00,197,90+ Days
5,2026-02-05,Sold,VIN20260005,10016,VXI,Petrol,Manual,2026,109,Maruti,Swift,Hatchback,Entry,3,Nashik Road,Nashik,MAHARASHTRA,720000.00,720000.00,193,90+ Days
6,2026-02-10,Available,VIN20260006,10017,ZXI+,Petrol,Automatic,2026,109,Maruti,Swift,Hatchback,Entry,3,Nashik Road,Nashik,MAHARASHTRA,850000.00,850000.00,188,90+ Days
7,2026-02-15,Sold,VIN20260007,10023,AX5,Petrol,Manual,2026,113,Mahindra,XUV 3XO,Suv,Mid,4,Nagpur Central,Nagpur,MAHARASHTRA,1200000.00,1200000.00,183,90+ Days
8,2026-02-20,Reserved,VIN20260008,10024,S11,Diesel,Manual,2026,114,Mahindra,Scorpio,Suv,Premium,4,Nagpur Central,Nagpur,MAHARASHTRA,1650000.00,1650000.00,178,90+ Days
9,2026-02-25,Sold,VIN20260009,10027,HTX,Petrol,Automatic,2026,117,Kia,Seltos,Suv,Mid,5,Thane West,Thane,MAHARASHTRA,1600000.00,1600000.00,173,90+ Days
10,2026-03-01,Available,VIN20260010,10028,GTX+,Petrol,Automatic,2026,118,Kia,Sonet,Suv,Entry,5,Thane West,Thane,MAHARASHTRA,1250000.00,1250000.00,169,90+ Days


In [0]:
# ============================================================
# FACT INVENTORY - VALIDATION
# ============================================================

print(
    "Total fact_inventory records:",
    df_fact_inventory.count()
)

print("Distinct inventory IDs:")

print(
    df_fact_inventory
    .select("inventory_id")
    .distinct()
    .count()
)

print("NULL inventory IDs:")

df_fact_inventory.filter(
    F.col("inventory_id").isNull()
).show()

print("Inventory status distribution:")

df_fact_inventory \
    .groupBy("status") \
    .count() \
    .orderBy("status") \
    .show()

Total fact_inventory records: 20
Distinct inventory IDs:
20
NULL inventory IDs:
+------------+-------------+------+---+----------+-------+---------+------------+------------------+--------+-----+----------+------------+-------+-----------+-------------+----+-----+-------------+---------------+--------------+----------------------+
|inventory_id|purchase_date|status|vin|vehicle_id|variant|fuel_type|transmission|manufacturing_year|model_id|brand|model_name|vehicle_type|segment|showroom_id|showroom_name|city|state|purchase_cost|inventory_value|inventory_days|inventory_age_category|
+------------+-------------+------+---+----------+-------+---------+------------+------------------+--------+-----+----------+------------+-------+-----------+-------------+----+-----+-------------+---------------+--------------+----------------------+
+------------+-------------+------+---+----------+-------+---------+------------+------------------+--------+-----+----------+------------+-------+-----------+--

In [0]:
# ============================================================
# FACT INVENTORY - WRITE TO GOLD
# ============================================================

df_fact_inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.fact_inventory"
    )

print(
    "SUCCESS: fact_inventory created in Gold."
)

SUCCESS: fact_inventory created in Gold.


In [0]:
# ============================================================
# FACT INVENTORY - VERIFY GOLD TABLE
# ============================================================

df_gold_fact_inventory = spark.table(
    "showroom_analytics.gold.fact_inventory"
)

display(df_gold_fact_inventory)

print(
    "Gold fact_inventory records:",
    df_gold_fact_inventory.count()
)

df_gold_fact_inventory.printSchema()

inventory_id,purchase_date,status,vin,vehicle_id,variant,fuel_type,transmission,manufacturing_year,model_id,brand,model_name,vehicle_type,segment,showroom_id,showroom_name,city,state,purchase_cost,inventory_value,inventory_days,inventory_age_category
1,2026-01-05,Sold,VIN20260001,10001,XZ+,Petrol,Manual,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,1050000.00,1050000.00,224,90+ Days
2,2026-01-12,Sold,VIN20260002,10002,XZ+ Lux,Petrol,Automatic,2026,101,Tata,Nexon,Suv,Mid,1,Pune Central,Pune,MAHARASHTRA,1200000.00,1200000.00,217,90+ Days
3,2026-01-18,Sold,VIN20260003,10008,S,Petrol,Manual,2026,105,Hyundai,Creta,Suv,Mid,2,Mumbai Andheri,Mumbai,MAHARASHTRA,1500000.00,1500000.00,211,90+ Days
4,2026-02-01,Available,VIN20260004,10009,SX,Petrol,Automatic,2026,105,Hyundai,Creta,Suv,Mid,2,Mumbai Andheri,Mumbai,MAHARASHTRA,1600000.00,1600000.00,197,90+ Days
5,2026-02-05,Sold,VIN20260005,10016,VXI,Petrol,Manual,2026,109,Maruti,Swift,Hatchback,Entry,3,Nashik Road,Nashik,MAHARASHTRA,720000.00,720000.00,193,90+ Days
6,2026-02-10,Available,VIN20260006,10017,ZXI+,Petrol,Automatic,2026,109,Maruti,Swift,Hatchback,Entry,3,Nashik Road,Nashik,MAHARASHTRA,850000.00,850000.00,188,90+ Days
7,2026-02-15,Sold,VIN20260007,10023,AX5,Petrol,Manual,2026,113,Mahindra,XUV 3XO,Suv,Mid,4,Nagpur Central,Nagpur,MAHARASHTRA,1200000.00,1200000.00,183,90+ Days
8,2026-02-20,Reserved,VIN20260008,10024,S11,Diesel,Manual,2026,114,Mahindra,Scorpio,Suv,Premium,4,Nagpur Central,Nagpur,MAHARASHTRA,1650000.00,1650000.00,178,90+ Days
9,2026-02-25,Sold,VIN20260009,10027,HTX,Petrol,Automatic,2026,117,Kia,Seltos,Suv,Mid,5,Thane West,Thane,MAHARASHTRA,1600000.00,1600000.00,173,90+ Days
10,2026-03-01,Available,VIN20260010,10028,GTX+,Petrol,Automatic,2026,118,Kia,Sonet,Suv,Entry,5,Thane West,Thane,MAHARASHTRA,1250000.00,1250000.00,169,90+ Days


Gold fact_inventory records: 20
root
 |-- inventory_id: long (nullable = true)
 |-- purchase_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- vin: string (nullable = true)
 |-- vehicle_id: integer (nullable = true)
 |-- variant: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- manufacturing_year: integer (nullable = true)
 |-- model_id: integer (nullable = true)
 |-- brand: string (nullable = true)
 |-- model_name: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- showroom_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- purchase_cost: decimal(18,2) (nullable = true)
 |-- inventory_value: decimal(18,2) (nullable = true)
 |-- inventory_days: integer (nullable = true)
 |-- inventory_age_category: string (nullable = true)



In [0]:
# ============================================================
# FACT EXPENSES - PREPARE SHOWROOM EXPENSES
# ============================================================

df_showroom_expenses = (
    df_expenses
    .select(
        F.col("expense_id").alias("expense_id"),
        F.col("showroom_id"),
        F.lit(None).cast("long").alias("campaign_id"),
        F.col("expense_date"),
        F.col("expense_type"),
        F.col("amount"),
        F.col("description")
    )
    .withColumn(
        "expense_category",
        F.lit("Showroom")
    )
)

In [0]:
# ============================================================
# FACT EXPENSES - PREPARE CAMPAIGN EXPENSES
# ============================================================

df_campaign_expenses = (
    df_campaign_expense
    .select(
        F.col("campaign_expense_id").alias("expense_id"),
        F.col("showroom_id"),
        F.col("campaign_id"),
        F.col("expense_date"),
        F.lit("Campaign").alias("expense_type"),
        F.col("amount"),
        F.lit("Campaign expense").alias("description")
    )
    .withColumn(
        "expense_category",
        F.lit("Campaign")
    )
)

In [0]:
# ============================================================
# FACT EXPENSES - COMBINE EXPENSE SOURCES
# ============================================================

df_fact_expenses = (
    df_showroom_expenses
    .unionByName(
        df_campaign_expenses
    )
)

display(df_fact_expenses)

expense_id,showroom_id,campaign_id,expense_date,expense_type,amount,description,expense_category
1,1,null,2026-01-31,Rent,180000.00,Monthly showroom rent,Showroom
2,1,null,2026-01-31,Electricity,32000.00,Monthly electricity bill,Showroom
3,2,null,2026-01-31,Marketing,75000.00,Digital marketing expense,Showroom
4,2,null,2026-02-28,Maintenance,28000.00,Showroom maintenance,Showroom
5,3,null,2026-02-28,Rent,150000.00,Monthly showroom rent,Showroom
6,3,null,2026-02-28,Salary,220000.00,Sales staff salary,Showroom
7,4,null,2026-03-31,Electricity,35000.00,Monthly electricity bill,Showroom
8,4,null,2026-03-31,Marketing,60000.00,Local marketing campaign,Showroom
9,5,null,2026-03-31,Rent,175000.00,Monthly showroom rent,Showroom
10,5,null,2026-04-30,Maintenance,24000.00,Vehicle display area maintenance,Showroom


In [0]:
# ============================================================
# FACT EXPENSES - BUSINESS ATTRIBUTES
# ============================================================

df_fact_expenses = (
    df_fact_expenses

    .withColumn(
        "expense_year",
        F.year("expense_date")
    )

    .withColumn(
        "expense_month",
        F.month("expense_date")
    )

    .withColumn(
        "expense_month_name",
        F.date_format(
            "expense_date",
            "MMMM"
        )
    )

    .withColumn(
        "expense_quarter",
        F.quarter("expense_date")
    )
)

display(df_fact_expenses)

expense_id,showroom_id,campaign_id,expense_date,expense_type,amount,description,expense_category,expense_year,expense_month,expense_month_name,expense_quarter
1,1,null,2026-01-31,Rent,180000.00,Monthly showroom rent,Showroom,2026,1,January,1
2,1,null,2026-01-31,Electricity,32000.00,Monthly electricity bill,Showroom,2026,1,January,1
3,2,null,2026-01-31,Marketing,75000.00,Digital marketing expense,Showroom,2026,1,January,1
4,2,null,2026-02-28,Maintenance,28000.00,Showroom maintenance,Showroom,2026,2,February,1
5,3,null,2026-02-28,Rent,150000.00,Monthly showroom rent,Showroom,2026,2,February,1
6,3,null,2026-02-28,Salary,220000.00,Sales staff salary,Showroom,2026,2,February,1
7,4,null,2026-03-31,Electricity,35000.00,Monthly electricity bill,Showroom,2026,3,March,1
8,4,null,2026-03-31,Marketing,60000.00,Local marketing campaign,Showroom,2026,3,March,1
9,5,null,2026-03-31,Rent,175000.00,Monthly showroom rent,Showroom,2026,3,March,1
10,5,null,2026-04-30,Maintenance,24000.00,Vehicle display area maintenance,Showroom,2026,4,April,2


In [0]:
# ============================================================
# FACT EXPENSES - VALIDATION
# ============================================================

print(
    "Total fact_expenses records:",
    df_fact_expenses.count()
)

print("NULL expense IDs:")

df_fact_expenses.filter(
    F.col("expense_id").isNull()
).show()

print("NULL showroom IDs:")

df_fact_expenses.filter(
    F.col("showroom_id").isNull()
).show()

print("Negative expense amounts:")

df_fact_expenses.filter(
    F.col("amount") < 0
).show()

print("Expense category distribution:")

df_fact_expenses \
    .groupBy("expense_category") \
    .count() \
    .show()

Total fact_expenses records: 30
NULL expense IDs:
+----------+-----------+-----------+------------+------------+------+-----------+----------------+------------+-------------+------------------+---------------+
|expense_id|showroom_id|campaign_id|expense_date|expense_type|amount|description|expense_category|expense_year|expense_month|expense_month_name|expense_quarter|
+----------+-----------+-----------+------------+------------+------+-----------+----------------+------------+-------------+------------------+---------------+
+----------+-----------+-----------+------------+------------+------+-----------+----------------+------------+-------------+------------------+---------------+

NULL showroom IDs:
+----------+-----------+-----------+------------+------------+------+-----------+----------------+------------+-------------+------------------+---------------+
|expense_id|showroom_id|campaign_id|expense_date|expense_type|amount|description|expense_category|expense_year|expense_month|

In [0]:
# ============================================================
# FACT EXPENSES - WRITE TO GOLD
# ============================================================

df_fact_expenses.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.fact_expenses"
    )

print(
    "SUCCESS: fact_expenses created in Gold."
)

SUCCESS: fact_expenses created in Gold.


In [0]:
# ============================================================
# FACT EXPENSES - VERIFY GOLD TABLE
# ============================================================

df_gold_fact_expenses = spark.table(
    "showroom_analytics.gold.fact_expenses"
)

display(df_gold_fact_expenses)

print(
    "Gold fact_expenses records:",
    df_gold_fact_expenses.count()
)

df_gold_fact_expenses.printSchema()

expense_id,showroom_id,campaign_id,expense_date,expense_type,amount,description,expense_category,expense_year,expense_month,expense_month_name,expense_quarter
1,1,null,2026-01-31,Rent,180000.00,Monthly showroom rent,Showroom,2026,1,January,1
2,1,null,2026-01-31,Electricity,32000.00,Monthly electricity bill,Showroom,2026,1,January,1
3,2,null,2026-01-31,Marketing,75000.00,Digital marketing expense,Showroom,2026,1,January,1
4,2,null,2026-02-28,Maintenance,28000.00,Showroom maintenance,Showroom,2026,2,February,1
5,3,null,2026-02-28,Rent,150000.00,Monthly showroom rent,Showroom,2026,2,February,1
6,3,null,2026-02-28,Salary,220000.00,Sales staff salary,Showroom,2026,2,February,1
7,4,null,2026-03-31,Electricity,35000.00,Monthly electricity bill,Showroom,2026,3,March,1
8,4,null,2026-03-31,Marketing,60000.00,Local marketing campaign,Showroom,2026,3,March,1
9,5,null,2026-03-31,Rent,175000.00,Monthly showroom rent,Showroom,2026,3,March,1
10,5,null,2026-04-30,Maintenance,24000.00,Vehicle display area maintenance,Showroom,2026,4,April,2


Gold fact_expenses records: 30
root
 |-- expense_id: long (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- campaign_id: long (nullable = true)
 |-- expense_date: date (nullable = true)
 |-- expense_type: string (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)
 |-- description: string (nullable = true)
 |-- expense_category: string (nullable = true)
 |-- expense_year: integer (nullable = true)
 |-- expense_month: integer (nullable = true)
 |-- expense_month_name: string (nullable = true)
 |-- expense_quarter: integer (nullable = true)



In [0]:
# ============================================================
# FACT LEADS - BUILD GOLD TABLE
# ============================================================
#
# Combines marketing leads with customer, showroom,
# salesperson and campaign information.
#
# This table will support:
# - Total leads
# - Qualified leads
# - Converted leads
# - Lost leads
# - Conversion rate
# - Lead source analysis
# - Campaign performance
# - Salesperson lead performance
# ============================================================

df_fact_leads = (
    df_leads.alias("l")

    # Customer
    .join(
        df_customer.alias("c"),
        F.col("l.customer_id") == F.col("c.customer_id"),
        "left"
    )

    # Showroom
    .join(
        df_showroom.alias("sh"),
        F.col("l.showroom_id") == F.col("sh.showroom_id"),
        "left"
    )

    # Salesperson
    .join(
        df_salesperson.alias("sp"),
        F.col("l.salesperson_id") == F.col("sp.salesperson_id"),
        "left"
    )

    # Campaign
    .join(
        df_campaign.alias("ca"),
        F.col("l.campaign_id") == F.col("ca.campaign_id"),
        "left"
    )

    .select(
        # Lead
        F.col("l.lead_id"),
        F.col("l.lead_date"),
        F.col("l.lead_source"),
        F.col("l.lead_status"),

        # Customer
        F.col("l.customer_id"),
        F.col("c.customer_name"),

        # Showroom
        F.col("l.showroom_id"),
        F.col("sh.showroom_name"),
        F.col("sh.city"),
        F.col("sh.state"),

        # Salesperson
        F.col("l.salesperson_id"),
        F.col("sp.salesperson_name"),

        # Campaign
        F.col("l.campaign_id"),
        F.col("ca.campaign_name"),
        F.col("ca.campaign_type"),
        F.col("ca.budget")
    )
)

display(df_fact_leads)

lead_id,lead_date,lead_source,lead_status,customer_id,customer_name,showroom_id,showroom_name,city,state,salesperson_id,salesperson_name,campaign_id,campaign_name,campaign_type,budget
1,2026-01-05,Google Ads,Converted,1,Sachin Kaware,1,Pune Central,Pune,MAHARASHTRA,1,Rahul Patil,2,Google Search Campaign,Digital,900000.00
2,2026-01-12,Instagram,Qualified,2,Amit Sharma,2,Mumbai Andheri,Mumbai,MAHARASHTRA,3,Priya Deshmukh,3,Instagram Suv Campaign,Social Media,700000.00
3,2026-01-15,Walk-in,New,3,Priya Deshmukh,3,Nashik Road,Nashik,MAHARASHTRA,5,Sneha Kulkarni,1,Diwali Mega Sale,Festival,1500000.00
4,2026-01-18,Google Ads,Contacted,4,Rahul Patil,1,Pune Central,Pune,MAHARASHTRA,2,Amit Sharma,4,New Year Offer,Festival,1200000.00
5,2026-01-22,Instagram,Qualified,5,Sneha Kulkarni,5,Thane West,Thane,MAHARASHTRA,9,Kunal Shinde,3,Instagram Suv Campaign,Social Media,700000.00
6,2026-01-25,Website,Converted,6,Rohit Joshi,2,Mumbai Andheri,Mumbai,MAHARASHTRA,4,Vikas Jadhav,5,Republic Day Offer,Festival,800000.00
7,2026-02-01,Facebook,Lost,7,Neha Patil,4,Nagpur Central,Nagpur,MAHARASHTRA,7,Sagar More,6,Summer Suv Sale,Seasonal,1100000.00
8,2026-02-05,Referral,Contacted,8,Kunal Shinde,5,Thane West,Thane,MAHARASHTRA,10,Pooja Pawar,7,Youtube Campaign,Digital,650000.00
9,2026-02-10,Google Ads,Qualified,9,Pooja Pawar,7,Aurangabad City,Aurangabad,MAHARASHTRA,13,Nitin Kadam,8,Exchange Bonus,Promotion,950000.00
10,2026-02-15,Website,Converted,10,Sagar More,3,Nashik Road,Nashik,MAHARASHTRA,6,Rohit Joshi,9,Monsoon Offer,Seasonal,850000.00


In [0]:
# ============================================================
# FACT LEADS - STANDARDIZE LEAD VALUES
# ============================================================

df_fact_leads = (
    df_fact_leads

    .withColumn(
        "lead_source",
        F.lower(F.trim(F.col("lead_source")))
    )

    .withColumn(
        "lead_source",
        F.initcap(F.col("lead_source"))
    )

    .withColumn(
        "lead_status",
        F.initcap(F.trim(F.col("lead_status")))
    )
)

display(
    df_fact_leads.select(
        "lead_source",
        "lead_status"
    ).distinct()
)

lead_source,lead_status
Google Ads,Converted
Instagram,Qualified
Walk-in,New
Google Ads,Contacted
Website,Converted
Facebook,Lost
Referral,Contacted
Google Ads,Qualified
Google Ads,New
Walk-in,Contacted


In [0]:
# ============================================================
# FACT LEADS - CONVERSION METRICS
# ============================================================

df_fact_leads = (
    df_fact_leads

    # Every lead counts as one
    .withColumn(
        "lead_count",
        F.lit(1)
    )

    # Converted lead indicator
    .withColumn(
        "converted_flag",
        F.when(
            F.col("lead_status") == "Converted",
            1
        ).otherwise(0)
    )

    # Qualified lead indicator
    .withColumn(
        "qualified_flag",
        F.when(
            F.col("lead_status") == "Qualified",
            1
        ).otherwise(0)
    )

    # Lost lead indicator
    .withColumn(
        "lost_flag",
        F.when(
            F.col("lead_status") == "Lost",
            1
        ).otherwise(0)
    )

    # Contacted lead indicator
    .withColumn(
        "contacted_flag",
        F.when(
            F.col("lead_status") == "Contacted",
            1
        ).otherwise(0)
    )
)

display(df_fact_leads)

lead_id,lead_date,lead_source,lead_status,customer_id,customer_name,showroom_id,showroom_name,city,state,salesperson_id,salesperson_name,campaign_id,campaign_name,campaign_type,budget,lead_count,converted_flag,qualified_flag,lost_flag,contacted_flag
1,2026-01-05,Google Ads,Converted,1,Sachin Kaware,1,Pune Central,Pune,MAHARASHTRA,1,Rahul Patil,2,Google Search Campaign,Digital,900000.00,1,1,0,0,0
2,2026-01-12,Instagram,Qualified,2,Amit Sharma,2,Mumbai Andheri,Mumbai,MAHARASHTRA,3,Priya Deshmukh,3,Instagram Suv Campaign,Social Media,700000.00,1,0,1,0,0
3,2026-01-15,Walk-in,New,3,Priya Deshmukh,3,Nashik Road,Nashik,MAHARASHTRA,5,Sneha Kulkarni,1,Diwali Mega Sale,Festival,1500000.00,1,0,0,0,0
4,2026-01-18,Google Ads,Contacted,4,Rahul Patil,1,Pune Central,Pune,MAHARASHTRA,2,Amit Sharma,4,New Year Offer,Festival,1200000.00,1,0,0,0,1
5,2026-01-22,Instagram,Qualified,5,Sneha Kulkarni,5,Thane West,Thane,MAHARASHTRA,9,Kunal Shinde,3,Instagram Suv Campaign,Social Media,700000.00,1,0,1,0,0
6,2026-01-25,Website,Converted,6,Rohit Joshi,2,Mumbai Andheri,Mumbai,MAHARASHTRA,4,Vikas Jadhav,5,Republic Day Offer,Festival,800000.00,1,1,0,0,0
7,2026-02-01,Facebook,Lost,7,Neha Patil,4,Nagpur Central,Nagpur,MAHARASHTRA,7,Sagar More,6,Summer Suv Sale,Seasonal,1100000.00,1,0,0,1,0
8,2026-02-05,Referral,Contacted,8,Kunal Shinde,5,Thane West,Thane,MAHARASHTRA,10,Pooja Pawar,7,Youtube Campaign,Digital,650000.00,1,0,0,0,1
9,2026-02-10,Google Ads,Qualified,9,Pooja Pawar,7,Aurangabad City,Aurangabad,MAHARASHTRA,13,Nitin Kadam,8,Exchange Bonus,Promotion,950000.00,1,0,1,0,0
10,2026-02-15,Website,Converted,10,Sagar More,3,Nashik Road,Nashik,MAHARASHTRA,6,Rohit Joshi,9,Monsoon Offer,Seasonal,850000.00,1,1,0,0,0


In [0]:
# ============================================================
# FACT LEADS - DATE ATTRIBUTES
# ============================================================

df_fact_leads = (
    df_fact_leads

    .withColumn(
        "lead_year",
        F.year("lead_date")
    )

    .withColumn(
        "lead_month",
        F.month("lead_date")
    )

    .withColumn(
        "lead_month_name",
        F.date_format(
            "lead_date",
            "MMMM"
        )
    )

    .withColumn(
        "lead_quarter",
        F.quarter("lead_date")
    )
)

display(df_fact_leads)

lead_id,lead_date,lead_source,lead_status,customer_id,customer_name,showroom_id,showroom_name,city,state,salesperson_id,salesperson_name,campaign_id,campaign_name,campaign_type,budget,lead_count,converted_flag,qualified_flag,lost_flag,contacted_flag,lead_year,lead_month,lead_month_name,lead_quarter
1,2026-01-05,Google Ads,Converted,1,Sachin Kaware,1,Pune Central,Pune,MAHARASHTRA,1,Rahul Patil,2,Google Search Campaign,Digital,900000.00,1,1,0,0,0,2026,1,January,1
2,2026-01-12,Instagram,Qualified,2,Amit Sharma,2,Mumbai Andheri,Mumbai,MAHARASHTRA,3,Priya Deshmukh,3,Instagram Suv Campaign,Social Media,700000.00,1,0,1,0,0,2026,1,January,1
3,2026-01-15,Walk-in,New,3,Priya Deshmukh,3,Nashik Road,Nashik,MAHARASHTRA,5,Sneha Kulkarni,1,Diwali Mega Sale,Festival,1500000.00,1,0,0,0,0,2026,1,January,1
4,2026-01-18,Google Ads,Contacted,4,Rahul Patil,1,Pune Central,Pune,MAHARASHTRA,2,Amit Sharma,4,New Year Offer,Festival,1200000.00,1,0,0,0,1,2026,1,January,1
5,2026-01-22,Instagram,Qualified,5,Sneha Kulkarni,5,Thane West,Thane,MAHARASHTRA,9,Kunal Shinde,3,Instagram Suv Campaign,Social Media,700000.00,1,0,1,0,0,2026,1,January,1
6,2026-01-25,Website,Converted,6,Rohit Joshi,2,Mumbai Andheri,Mumbai,MAHARASHTRA,4,Vikas Jadhav,5,Republic Day Offer,Festival,800000.00,1,1,0,0,0,2026,1,January,1
7,2026-02-01,Facebook,Lost,7,Neha Patil,4,Nagpur Central,Nagpur,MAHARASHTRA,7,Sagar More,6,Summer Suv Sale,Seasonal,1100000.00,1,0,0,1,0,2026,2,February,1
8,2026-02-05,Referral,Contacted,8,Kunal Shinde,5,Thane West,Thane,MAHARASHTRA,10,Pooja Pawar,7,Youtube Campaign,Digital,650000.00,1,0,0,0,1,2026,2,February,1
9,2026-02-10,Google Ads,Qualified,9,Pooja Pawar,7,Aurangabad City,Aurangabad,MAHARASHTRA,13,Nitin Kadam,8,Exchange Bonus,Promotion,950000.00,1,0,1,0,0,2026,2,February,1
10,2026-02-15,Website,Converted,10,Sagar More,3,Nashik Road,Nashik,MAHARASHTRA,6,Rohit Joshi,9,Monsoon Offer,Seasonal,850000.00,1,1,0,0,0,2026,2,February,1


In [0]:
# ============================================================
# FACT LEADS - VALIDATION
# ============================================================

print(
    "Total fact_leads records:",
    df_fact_leads.count()
)

print("Distinct lead IDs:")

print(
    df_fact_leads
    .select("lead_id")
    .distinct()
    .count()
)

print("NULL lead IDs:")

df_fact_leads.filter(
    F.col("lead_id").isNull()
).show()

print("Lead status distribution:")

df_fact_leads \
    .groupBy("lead_status") \
    .count() \
    .orderBy("lead_status") \
    .show()

print("Lead source distribution:")

df_fact_leads \
    .groupBy("lead_source") \
    .count() \
    .orderBy("lead_source") \
    .show()

Total fact_leads records: 20
Distinct lead IDs:
20
NULL lead IDs:
+-------+---------+-----------+-----------+-----------+-------------+-----------+-------------+----+-----+--------------+----------------+-----------+-------------+-------------+------+----------+--------------+--------------+---------+--------------+---------+----------+---------------+------------+
|lead_id|lead_date|lead_source|lead_status|customer_id|customer_name|showroom_id|showroom_name|city|state|salesperson_id|salesperson_name|campaign_id|campaign_name|campaign_type|budget|lead_count|converted_flag|qualified_flag|lost_flag|contacted_flag|lead_year|lead_month|lead_month_name|lead_quarter|
+-------+---------+-----------+-----------+-----------+-------------+-----------+-------------+----+-----+--------------+----------------+-----------+-------------+-------------+------+----------+--------------+--------------+---------+--------------+---------+----------+---------------+------------+
+-------+---------+---------

In [0]:
# ============================================================
# FACT LEADS - WRITE TO GOLD
# ============================================================

df_fact_leads.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.fact_leads"
    )

print(
    "SUCCESS: fact_leads created in Gold."
)

SUCCESS: fact_leads created in Gold.


In [0]:
# ============================================================
# FACT LEADS - VERIFY GOLD TABLE
# ============================================================

df_gold_fact_leads = spark.table(
    "showroom_analytics.gold.fact_leads"
)

display(df_gold_fact_leads)

print(
    "Gold fact_leads records:",
    df_gold_fact_leads.count()
)

df_gold_fact_leads.printSchema()

lead_id,lead_date,lead_source,lead_status,customer_id,customer_name,showroom_id,showroom_name,city,state,salesperson_id,salesperson_name,campaign_id,campaign_name,campaign_type,budget,lead_count,converted_flag,qualified_flag,lost_flag,contacted_flag,lead_year,lead_month,lead_month_name,lead_quarter
1,2026-01-05,Google Ads,Converted,1,Sachin Kaware,1,Pune Central,Pune,MAHARASHTRA,1,Rahul Patil,2,Google Search Campaign,Digital,900000.00,1,1,0,0,0,2026,1,January,1
2,2026-01-12,Instagram,Qualified,2,Amit Sharma,2,Mumbai Andheri,Mumbai,MAHARASHTRA,3,Priya Deshmukh,3,Instagram Suv Campaign,Social Media,700000.00,1,0,1,0,0,2026,1,January,1
3,2026-01-15,Walk-in,New,3,Priya Deshmukh,3,Nashik Road,Nashik,MAHARASHTRA,5,Sneha Kulkarni,1,Diwali Mega Sale,Festival,1500000.00,1,0,0,0,0,2026,1,January,1
4,2026-01-18,Google Ads,Contacted,4,Rahul Patil,1,Pune Central,Pune,MAHARASHTRA,2,Amit Sharma,4,New Year Offer,Festival,1200000.00,1,0,0,0,1,2026,1,January,1
5,2026-01-22,Instagram,Qualified,5,Sneha Kulkarni,5,Thane West,Thane,MAHARASHTRA,9,Kunal Shinde,3,Instagram Suv Campaign,Social Media,700000.00,1,0,1,0,0,2026,1,January,1
6,2026-01-25,Website,Converted,6,Rohit Joshi,2,Mumbai Andheri,Mumbai,MAHARASHTRA,4,Vikas Jadhav,5,Republic Day Offer,Festival,800000.00,1,1,0,0,0,2026,1,January,1
7,2026-02-01,Facebook,Lost,7,Neha Patil,4,Nagpur Central,Nagpur,MAHARASHTRA,7,Sagar More,6,Summer Suv Sale,Seasonal,1100000.00,1,0,0,1,0,2026,2,February,1
8,2026-02-05,Referral,Contacted,8,Kunal Shinde,5,Thane West,Thane,MAHARASHTRA,10,Pooja Pawar,7,Youtube Campaign,Digital,650000.00,1,0,0,0,1,2026,2,February,1
9,2026-02-10,Google Ads,Qualified,9,Pooja Pawar,7,Aurangabad City,Aurangabad,MAHARASHTRA,13,Nitin Kadam,8,Exchange Bonus,Promotion,950000.00,1,0,1,0,0,2026,2,February,1
10,2026-02-15,Website,Converted,10,Sagar More,3,Nashik Road,Nashik,MAHARASHTRA,6,Rohit Joshi,9,Monsoon Offer,Seasonal,850000.00,1,1,0,0,0,2026,2,February,1


Gold fact_leads records: 20
root
 |-- lead_id: long (nullable = true)
 |-- lead_date: date (nullable = true)
 |-- lead_source: string (nullable = true)
 |-- lead_status: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- showroom_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- salesperson_id: integer (nullable = true)
 |-- salesperson_name: string (nullable = true)
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_type: string (nullable = true)
 |-- budget: decimal(18,2) (nullable = true)
 |-- lead_count: integer (nullable = true)
 |-- converted_flag: integer (nullable = true)
 |-- qualified_flag: integer (nullable = true)
 |-- lost_flag: integer (nullable = true)
 |-- contacted_flag: integer (nullable = true)
 |-- lead_year: integer (nullable = true)
 |-- lead_month

In [0]:
# ============================================================
# DIM VEHICLE MODEL - BUILD GOLD TABLE
# ============================================================

df_dim_vehicle_model = (
    df_vehicle_model
    .select(
        "model_id",
        "brand",
        "model_name",
        "vehicle_type",
        "segment"
    )
    .dropDuplicates(["model_id"])
)

display(df_dim_vehicle_model)

model_id,brand,model_name,vehicle_type,segment
101,Tata,Nexon,Suv,Mid
102,Tata,Punch,Suv,Entry
103,Tata,Harrier,Suv,Premium
104,Tata,Safari,Suv,Premium
105,Hyundai,Creta,Suv,Mid
106,Hyundai,Venue,Suv,Entry
107,Hyundai,Verna,Sedan,Mid
108,Hyundai,i20,Hatchback,Entry
109,Maruti,Swift,Hatchback,Entry
110,Maruti,Baleno,Hatchback,Mid


In [0]:
# ============================================================
# DIM VEHICLE MODEL - VALIDATION
# ============================================================

print(
    "Total vehicle models:",
    df_dim_vehicle_model.count()
)

print("NULL model IDs:")

df_dim_vehicle_model.filter(
    F.col("model_id").isNull()
).show()

print("Duplicate model IDs:")

df_dim_vehicle_model \
    .groupBy("model_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

Total vehicle models: 28
NULL model IDs:
+--------+-----+----------+------------+-------+
|model_id|brand|model_name|vehicle_type|segment|
+--------+-----+----------+------------+-------+
+--------+-----+----------+------------+-------+

Duplicate model IDs:
+--------+-----+
|model_id|count|
+--------+-----+
+--------+-----+



In [0]:
# ============================================================
# DIM VEHICLE MODEL - WRITE TO GOLD
# ============================================================

df_dim_vehicle_model.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.dim_vehicle_model"
    )

print(
    "SUCCESS: dim_vehicle_model created in Gold."
)

SUCCESS: dim_vehicle_model created in Gold.


In [0]:
# ============================================================
# DIM VEHICLE MODEL - VERIFY
# ============================================================

df_gold_dim_vehicle_model = spark.table(
    "showroom_analytics.gold.dim_vehicle_model"
)

display(df_gold_dim_vehicle_model)

print(
    "Gold dim_vehicle_model records:",
    df_gold_dim_vehicle_model.count()
)

model_id,brand,model_name,vehicle_type,segment
105,Hyundai,Creta,Suv,Mid
121,Toyota,Fortuner,Suv,Luxury
104,Tata,Safari,Suv,Premium
108,Hyundai,i20,Hatchback,Entry
111,Maruti,Brezza,Suv,Mid
123,Honda,City,Sedan,Mid
124,Honda,Elevate,Suv,Mid
109,Maruti,Swift,Hatchback,Entry
115,Mahindra,XUV700,Suv,Premium
126,Skoda,Slavia,Sedan,Mid


Gold dim_vehicle_model records: 28


In [0]:
# ============================================================
# DIM VEHICLE - BUILD GOLD TABLE
# ============================================================

df_dim_vehicle = (
    df_vehicle.alias("v")
    .join(
        df_vehicle_model.alias("vm"),
        F.col("v.model_id") == F.col("vm.model_id"),
        "left"
    )
    .select(
        F.col("v.vehicle_id"),
        F.col("v.model_id"),
        F.col("vm.brand"),
        F.col("vm.model_name"),
        F.col("v.variant"),
        F.col("vm.vehicle_type"),
        F.col("vm.segment"),
        F.col("v.fuel_type"),
        F.col("v.transmission"),
        F.col("v.manufacturing_year"),
        F.col("v.base_price")
    )
    .dropDuplicates(["vehicle_id"])
)

display(df_dim_vehicle)

vehicle_id,model_id,brand,model_name,variant,vehicle_type,segment,fuel_type,transmission,manufacturing_year,base_price
10002,101,Tata,Nexon,XZ+ Lux,Suv,Mid,Petrol,Automatic,2026,1400000.000000000000000000
10022,112,Maruti,Grand Vitara,Alpha+,Suv,Premium,Hybrid,Automatic,2026,1900000.000000000000000000
10003,101,Tata,Nexon,XZ+ Diesel,Suv,Mid,Diesel,Manual,2026,1450000.000000000000000000
10020,111,Maruti,Brezza,ZXI,Suv,Mid,Petrol,Manual,2026,1150000.000000000000000000
10031,121,Toyota,Fortuner,Legender,Suv,Luxury,Diesel,Automatic,2026,4800000.000000000000000000
10014,107,Hyundai,Verna,SX Turbo,Sedan,Mid,Petrol,Automatic,2026,1650000.000000000000000000
10038,128,Volkswagen,Virtus,GT,Sedan,Mid,Petrol,Automatic,2026,1850000.000000000000000000
10001,101,Tata,Nexon,XZ+,Suv,Mid,Petrol,Manual,2026,1250000.000000000000000000
10008,105,Hyundai,Creta,S,Suv,Mid,Petrol,Manual,2026,1450000.000000000000000000
10032,122,Toyota,Urban Cruiser,High,Suv,Mid,Petrol,Automatic,2026,1600000.000000000000000000


In [0]:
# ============================================================
# DIM VEHICLE - VALIDATION
# ============================================================

print(
    "Total vehicles:",
    df_dim_vehicle.count()
)

print("NULL vehicle IDs:")

df_dim_vehicle.filter(
    F.col("vehicle_id").isNull()
).show()

print("Duplicate vehicle IDs:")

df_dim_vehicle \
    .groupBy("vehicle_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

print("Vehicles with missing model information:")

df_dim_vehicle.filter(
    F.col("model_id").isNull()
).show()

Total vehicles: 38
NULL vehicle IDs:
+----------+--------+-----+----------+-------+------------+-------+---------+------------+------------------+----------+
|vehicle_id|model_id|brand|model_name|variant|vehicle_type|segment|fuel_type|transmission|manufacturing_year|base_price|
+----------+--------+-----+----------+-------+------------+-------+---------+------------+------------------+----------+
+----------+--------+-----+----------+-------+------------+-------+---------+------------+------------------+----------+

Duplicate vehicle IDs:
+----------+-----+
|vehicle_id|count|
+----------+-----+
+----------+-----+

Vehicles with missing model information:
+----------+--------+-----+----------+-------+------------+-------+---------+------------+------------------+----------+
|vehicle_id|model_id|brand|model_name|variant|vehicle_type|segment|fuel_type|transmission|manufacturing_year|base_price|
+----------+--------+-----+----------+-------+------------+-------+---------+------------+-----

In [0]:
# ============================================================
# DIM VEHICLE - WRITE TO GOLD
# ============================================================

df_dim_vehicle.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.dim_vehicle"
    )

print(
    "SUCCESS: dim_vehicle created in Gold."
)

SUCCESS: dim_vehicle created in Gold.


In [0]:
# ============================================================
# DIM VEHICLE - VERIFY
# ============================================================

df_gold_dim_vehicle = spark.table(
    "showroom_analytics.gold.dim_vehicle"
)

display(df_gold_dim_vehicle)

print(
    "Gold dim_vehicle records:",
    df_gold_dim_vehicle.count()
)

df_gold_dim_vehicle.printSchema()

vehicle_id,model_id,brand,model_name,variant,vehicle_type,segment,fuel_type,transmission,manufacturing_year,base_price
10002,101,Tata,Nexon,XZ+ Lux,Suv,Mid,Petrol,Automatic,2026,1400000.000000000000000000
10022,112,Maruti,Grand Vitara,Alpha+,Suv,Premium,Hybrid,Automatic,2026,1900000.000000000000000000
10003,101,Tata,Nexon,XZ+ Diesel,Suv,Mid,Diesel,Manual,2026,1450000.000000000000000000
10020,111,Maruti,Brezza,ZXI,Suv,Mid,Petrol,Manual,2026,1150000.000000000000000000
10031,121,Toyota,Fortuner,Legender,Suv,Luxury,Diesel,Automatic,2026,4800000.000000000000000000
10014,107,Hyundai,Verna,SX Turbo,Sedan,Mid,Petrol,Automatic,2026,1650000.000000000000000000
10038,128,Volkswagen,Virtus,GT,Sedan,Mid,Petrol,Automatic,2026,1850000.000000000000000000
10001,101,Tata,Nexon,XZ+,Suv,Mid,Petrol,Manual,2026,1250000.000000000000000000
10008,105,Hyundai,Creta,S,Suv,Mid,Petrol,Manual,2026,1450000.000000000000000000
10032,122,Toyota,Urban Cruiser,High,Suv,Mid,Petrol,Automatic,2026,1600000.000000000000000000


Gold dim_vehicle records: 38
root
 |-- vehicle_id: integer (nullable = true)
 |-- model_id: integer (nullable = true)
 |-- brand: string (nullable = true)
 |-- model_name: string (nullable = true)
 |-- variant: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- fuel_type: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- manufacturing_year: integer (nullable = true)
 |-- base_price: decimal(38,18) (nullable = true)



In [0]:
# ============================================================
# DIM SHOWROOM - BUILD GOLD TABLE
# ============================================================

df_dim_showroom = (
    df_showroom
    .select(
        "showroom_id",
        "showroom_name",
        "brand",
        "city",
        "state",
        "manager_name",
        "opening_date"
    )
    .dropDuplicates(["showroom_id"])
)

display(df_dim_showroom)

showroom_id,showroom_name,brand,city,state,manager_name,opening_date
1,Pune Central,Tata,Pune,MAHARASHTRA,Rahul Patil,2020-01-15
2,Mumbai Andheri,Hyundai,Mumbai,MAHARASHTRA,Amit Sharma,2019-06-10
3,Nashik Road,Maruti,Nashik,MAHARASHTRA,Priya Deshmukh,2021-03-20
4,Nagpur Central,Mahindra,Nagpur,MAHARASHTRA,Vikas Jadhav,2020-08-12
5,Thane West,Kia,Thane,MAHARASHTRA,Sneha Kulkarni,2022-01-05
6,Navi Mumbai,Toyota,Navi Mumbai,MAHARASHTRA,Rohit Joshi,2021-09-18
7,Aurangabad City,Honda,Aurangabad,MAHARASHTRA,Sagar More,2020-11-25
8,Kolhapur Central,Skoda,Kolhapur,MAHARASHTRA,Neha Patil,2022-04-15
9,Ahmednagar Road,Volkswagen,Ahmednagar,MAHARASHTRA,Kunal Shinde,2021-07-10
10,Solapur Central,Tata,Solapur,MAHARASHTRA,Pooja Pawar,2022-06-20


In [0]:
# ============================================================
# DIM SHOWROOM - VALIDATION
# ============================================================

print(
    "Total showrooms:",
    df_dim_showroom.count()
)

print("NULL showroom IDs:")

df_dim_showroom.filter(
    F.col("showroom_id").isNull()
).show()

print("Duplicate showroom IDs:")

df_dim_showroom \
    .groupBy("showroom_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

Total showrooms: 15
NULL showroom IDs:
+-----------+-------------+-----+----+-----+------------+------------+
|showroom_id|showroom_name|brand|city|state|manager_name|opening_date|
+-----------+-------------+-----+----+-----+------------+------------+
+-----------+-------------+-----+----+-----+------------+------------+

Duplicate showroom IDs:
+-----------+-----+
|showroom_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
# ============================================================
# DIM SHOWROOM - WRITE TO GOLD
# ============================================================

df_dim_showroom.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.dim_showroom"
    )

print(
    "SUCCESS: dim_showroom created in Gold."
)

SUCCESS: dim_showroom created in Gold.


In [0]:
# ============================================================
# DIM SHOWROOM - VERIFY
# ============================================================

df_gold_dim_showroom = spark.table(
    "showroom_analytics.gold.dim_showroom"
)

display(df_gold_dim_showroom)

print(
    "Gold dim_showroom records:",
    df_gold_dim_showroom.count()
)

df_gold_dim_showroom.printSchema()

showroom_id,showroom_name,brand,city,state,manager_name,opening_date
12,Borivali Mumbai,Maruti,Mumbai,MAHARASHTRA,Meena Shah,2020-05-15
5,Thane West,Kia,Thane,MAHARASHTRA,Sneha Kulkarni,2022-01-05
10,Solapur Central,Tata,Solapur,MAHARASHTRA,Pooja Pawar,2022-06-20
1,Pune Central,Tata,Pune,MAHARASHTRA,Rahul Patil,2020-01-15
3,Nashik Road,Maruti,Nashik,MAHARASHTRA,Priya Deshmukh,2021-03-20
2,Mumbai Andheri,Hyundai,Mumbai,MAHARASHTRA,Amit Sharma,2019-06-10
13,Kalyan East,Mahindra,Kalyan,MAHARASHTRA,Nitin Kadam,2023-02-10
14,Satara Road,Kia,Satara,MAHARASHTRA,Riya Chavan,2023-05-18
6,Navi Mumbai,Toyota,Navi Mumbai,MAHARASHTRA,Rohit Joshi,2021-09-18
9,Ahmednagar Road,Volkswagen,Ahmednagar,MAHARASHTRA,Kunal Shinde,2021-07-10


Gold dim_showroom records: 15
root
 |-- showroom_id: integer (nullable = true)
 |-- showroom_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- manager_name: string (nullable = true)
 |-- opening_date: date (nullable = true)



In [0]:
# ============================================================
# DIM SALESPERSON - BUILD GOLD TABLE
# ============================================================

df_dim_salesperson = (
    df_salesperson
    .select(
        "salesperson_id",
        "salesperson_name",
        "showroom_id",
        "joining_date"
    )
    .dropDuplicates(["salesperson_id"])
)

display(df_dim_salesperson)

salesperson_id,salesperson_name,showroom_id,joining_date
1,Rahul Patil,1,2022-04-10
2,Amit Sharma,1,2023-01-15
3,Priya Deshmukh,2,2021-07-20
4,Vikas Jadhav,2,2023-03-12
5,Sneha Kulkarni,3,2022-09-05
6,Rohit Joshi,3,2024-01-18
7,Sagar More,4,2021-11-25
8,Neha Patil,4,2023-06-10
9,Kunal Shinde,5,2022-02-14
10,Pooja Pawar,5,2024-04-01


In [0]:
# ============================================================
# DIM SALESPERSON - VALIDATION
# ============================================================

print(
    "Total salespersons:",
    df_dim_salesperson.count()
)

print("NULL salesperson IDs:")

df_dim_salesperson.filter(
    F.col("salesperson_id").isNull()
).show()

print("Duplicate salesperson IDs:")

df_dim_salesperson \
    .groupBy("salesperson_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

Total salespersons: 15
NULL salesperson IDs:
+--------------+----------------+-----------+------------+
|salesperson_id|salesperson_name|showroom_id|joining_date|
+--------------+----------------+-----------+------------+
+--------------+----------------+-----------+------------+

Duplicate salesperson IDs:
+--------------+-----+
|salesperson_id|count|
+--------------+-----+
+--------------+-----+



In [0]:
# ============================================================
# DIM SALESPERSON - WRITE TO GOLD
# ============================================================

df_dim_salesperson.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.dim_salesperson"
    )

print(
    "SUCCESS: dim_salesperson created in Gold."
)

SUCCESS: dim_salesperson created in Gold.


In [0]:
# ============================================================
# DIM SALESPERSON - VERIFY
# ============================================================

df_gold_dim_salesperson = spark.table(
    "showroom_analytics.gold.dim_salesperson"
)

display(df_gold_dim_salesperson)

print(
    "Gold dim_salesperson records:",
    df_gold_dim_salesperson.count()
)

df_gold_dim_salesperson.printSchema()

salesperson_id,salesperson_name,showroom_id,joining_date
12,Meena Shah,6,2023-10-22
5,Sneha Kulkarni,3,2022-09-05
10,Pooja Pawar,5,2024-04-01
1,Rahul Patil,1,2022-04-10
3,Priya Deshmukh,2,2021-07-20
2,Amit Sharma,1,2023-01-15
13,Nitin Kadam,7,2021-05-17
14,Riya Chavan,8,2024-02-10
6,Rohit Joshi,3,2024-01-18
9,Kunal Shinde,5,2022-02-14


Gold dim_salesperson records: 15
root
 |-- salesperson_id: integer (nullable = true)
 |-- salesperson_name: string (nullable = true)
 |-- showroom_id: integer (nullable = true)
 |-- joining_date: date (nullable = true)



In [0]:
# ============================================================
# DIM CUSTOMER - BUILD GOLD TABLE
# ============================================================

df_dim_customer = (
    df_customer
    .select(
        "customer_id",
        "customer_name",
        "city",
        "age",
        "gender",
        "occupation"
    )
    .dropDuplicates(["customer_id"])
)

display(df_dim_customer)

customer_id,customer_name,city,age,gender,occupation
1,Sachin Kaware,Pune,27,Male,Engineer
2,Amit Sharma,Mumbai,32,Male,Business
3,Priya Deshmukh,Nashik,29,Female,Teacher
4,Rahul Patil,Pune,35,Male,Business
5,Sneha Kulkarni,Thane,28,Female,Engineer
6,Rohit Joshi,Mumbai,41,Male,Doctor
7,Neha Patil,Nagpur,31,Female,Government
8,Kunal Shinde,Kolhapur,26,Male,Engineer
9,Pooja Pawar,Aurangabad,34,Female,Teacher
10,Sagar More,Nashik,39,Male,Business


In [0]:
# ============================================================
# DIM CUSTOMER - VALIDATION
# ============================================================

print(
    "Total customers:",
    df_dim_customer.count()
)

print("NULL customer IDs:")

df_dim_customer.filter(
    F.col("customer_id").isNull()
).show()

print("Duplicate customer IDs:")

df_dim_customer \
    .groupBy("customer_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

Total customers: 20
NULL customer IDs:
+-----------+-------------+----+---+------+----------+
|customer_id|customer_name|city|age|gender|occupation|
+-----------+-------------+----+---+------+----------+
+-----------+-------------+----+---+------+----------+

Duplicate customer IDs:
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
# ============================================================
# DIM CUSTOMER - WRITE TO GOLD
# ============================================================

df_dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.dim_customer"
    )

print(
    "SUCCESS: dim_customer created in Gold."
)

SUCCESS: dim_customer created in Gold.


In [0]:
# ============================================================
# DIM CUSTOMER - VERIFY
# ============================================================

df_gold_dim_customer = spark.table(
    "showroom_analytics.gold.dim_customer"
)

display(df_gold_dim_customer)

print(
    "Gold dim_customer records:",
    df_gold_dim_customer.count()
)

df_gold_dim_customer.printSchema()

customer_id,customer_name,city,age,gender,occupation
9,Pooja Pawar,Aurangabad,34,Female,Teacher
11,Meena Shah,Mumbai,45,Female,Business
14,Riya Chavan,Satara,25,Female,null
17,Vivek Sharma,Thane,36,Male,Private
3,Priya Deshmukh,Nashik,29,Female,Teacher
4,Rahul Patil,Pune,35,Male,Business
19,Rakesh Jadhav,Pune,40,Male,Business
6,Rohit Joshi,Mumbai,41,Male,Doctor
13,Vikas Jadhav,Nagpur,37,Male,Government
20,Snehal Joshi,Mumbai,27,Female,Engineer


Gold dim_customer records: 20
root
 |-- customer_id: long (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- occupation: string (nullable = true)



In [0]:
# ============================================================
# DIM CAMPAIGN - BUILD GOLD TABLE
# ============================================================

df_dim_campaign = (
    df_campaign
    .select(
        "campaign_id",
        "campaign_name",
        "campaign_type",
        "start_date",
        "end_date",
        "budget"
    )
    .dropDuplicates(["campaign_id"])
)

display(df_dim_campaign)

campaign_id,campaign_name,campaign_type,start_date,end_date,budget
1,Diwali Mega Sale,Festival,2025-10-01,2025-11-15,1500000.00
2,Google Search Campaign,Digital,2025-11-01,2025-12-31,900000.00
3,Instagram Suv Campaign,Social Media,2025-12-01,2026-01-31,700000.00
4,New Year Offer,Festival,2026-01-01,2026-01-31,1200000.00
5,Republic Day Offer,Festival,2026-01-15,2026-01-31,800000.00
6,Summer Suv Sale,Seasonal,2026-03-01,2026-04-30,1100000.00
7,Youtube Campaign,Digital,2026-03-15,2026-05-15,650000.00
8,Exchange Bonus,Promotion,2026-04-01,2026-05-31,950000.00
9,Monsoon Offer,Seasonal,2026-06-01,2026-07-31,850000.00
10,Independence Sale,Festival,2026-08-01,2026-08-20,1000000.00


In [0]:
# ============================================================
# DIM CAMPAIGN - BUSINESS ATTRIBUTES
# ============================================================

df_dim_campaign = (
    df_dim_campaign
    .withColumn(
        "campaign_duration_days",
        F.datediff(
            F.col("end_date"),
            F.col("start_date")
        ) + 1
    )
)

display(df_dim_campaign)

campaign_id,campaign_name,campaign_type,start_date,end_date,budget,campaign_duration_days
1,Diwali Mega Sale,Festival,2025-10-01,2025-11-15,1500000.00,46
2,Google Search Campaign,Digital,2025-11-01,2025-12-31,900000.00,61
3,Instagram Suv Campaign,Social Media,2025-12-01,2026-01-31,700000.00,62
4,New Year Offer,Festival,2026-01-01,2026-01-31,1200000.00,31
5,Republic Day Offer,Festival,2026-01-15,2026-01-31,800000.00,17
6,Summer Suv Sale,Seasonal,2026-03-01,2026-04-30,1100000.00,61
7,Youtube Campaign,Digital,2026-03-15,2026-05-15,650000.00,62
8,Exchange Bonus,Promotion,2026-04-01,2026-05-31,950000.00,61
9,Monsoon Offer,Seasonal,2026-06-01,2026-07-31,850000.00,61
10,Independence Sale,Festival,2026-08-01,2026-08-20,1000000.00,20


In [0]:
# ============================================================
# DIM CAMPAIGN - VALIDATION
# ============================================================

print(
    "Total campaigns:",
    df_dim_campaign.count()
)

print("NULL campaign IDs:")

df_dim_campaign.filter(
    F.col("campaign_id").isNull()
).show()

print("Duplicate campaign IDs:")

df_dim_campaign \
    .groupBy("campaign_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

Total campaigns: 10
NULL campaign IDs:
+-----------+-------------+-------------+----------+--------+------+----------------------+
|campaign_id|campaign_name|campaign_type|start_date|end_date|budget|campaign_duration_days|
+-----------+-------------+-------------+----------+--------+------+----------------------+
+-----------+-------------+-------------+----------+--------+------+----------------------+

Duplicate campaign IDs:
+-----------+-----+
|campaign_id|count|
+-----------+-----+
+-----------+-----+



In [0]:
# ============================================================
# DIM CAMPAIGN - WRITE TO GOLD
# ============================================================

df_dim_campaign.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.dim_campaign"
    )

print(
    "SUCCESS: dim_campaign created in Gold."
)

SUCCESS: dim_campaign created in Gold.


In [0]:
# ============================================================
# DIM CAMPAIGN - VERIFY
# ============================================================

df_gold_dim_campaign = spark.table(
    "showroom_analytics.gold.dim_campaign"
)

display(df_gold_dim_campaign)

print(
    "Gold dim_campaign records:",
    df_gold_dim_campaign.count()
)

df_gold_dim_campaign.printSchema()

campaign_id,campaign_name,campaign_type,start_date,end_date,budget,campaign_duration_days
5,Republic Day Offer,Festival,2026-01-15,2026-01-31,800000.00,17
10,Independence Sale,Festival,2026-08-01,2026-08-20,1000000.00,20
1,Diwali Mega Sale,Festival,2025-10-01,2025-11-15,1500000.00,46
3,Instagram Suv Campaign,Social Media,2025-12-01,2026-01-31,700000.00,62
2,Google Search Campaign,Digital,2025-11-01,2025-12-31,900000.00,61
6,Summer Suv Sale,Seasonal,2026-03-01,2026-04-30,1100000.00,61
9,Monsoon Offer,Seasonal,2026-06-01,2026-07-31,850000.00,61
7,Youtube Campaign,Digital,2026-03-15,2026-05-15,650000.00,62
4,New Year Offer,Festival,2026-01-01,2026-01-31,1200000.00,31
8,Exchange Bonus,Promotion,2026-04-01,2026-05-31,950000.00,61


Gold dim_campaign records: 10
root
 |-- campaign_id: integer (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- campaign_type: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- budget: decimal(18,2) (nullable = true)
 |-- campaign_duration_days: integer (nullable = true)



In [0]:
# ============================================================
# DIM DATE - DETERMINE DATE RANGE
# ============================================================

sales_min_date = df_sales.select(
    F.min("sale_date")
).collect()[0][0]

sales_max_date = df_sales.select(
    F.max("sale_date")
).collect()[0][0]

lead_min_date = df_leads.select(
    F.min("lead_date")
).collect()[0][0]

lead_max_date = df_leads.select(
    F.max("lead_date")
).collect()[0][0]

inventory_min_date = df_inventory.select(
    F.min("purchase_date")
).collect()[0][0]

inventory_max_date = df_inventory.select(
    F.max("purchase_date")
).collect()[0][0]

expense_min_date = df_expenses.select(
    F.min("expense_date")
).collect()[0][0]

expense_max_date = df_expenses.select(
    F.max("expense_date")
).collect()[0][0]

print("Sales:", sales_min_date, "to", sales_max_date)
print("Leads:", lead_min_date, "to", lead_max_date)
print("Inventory:", inventory_min_date, "to", inventory_max_date)
print("Expenses:", expense_min_date, "to", expense_max_date)

Sales: 2026-01-10 to 2026-04-25
Leads: 2026-01-05 to 2026-04-05
Inventory: 2026-01-05 to 2026-04-20
Expenses: 2026-01-31 to 2026-06-30


In [0]:
# ============================================================
# DIM DATE - CREATE COMPLETE DATE RANGE
# ============================================================

from functools import reduce

min_dates = [
    sales_min_date,
    lead_min_date,
    inventory_min_date,
    expense_min_date
]

max_dates = [
    sales_max_date,
    lead_max_date,
    inventory_max_date,
    expense_max_date
]

project_start_date = min(
    date_value for date_value in min_dates
    if date_value is not None
)

project_end_date = max(
    date_value for date_value in max_dates
    if date_value is not None
)

print("Project start date:", project_start_date)
print("Project end date:", project_end_date)

Project start date: 2026-01-05
Project end date: 2026-06-30


In [0]:
# ============================================================
# DIM DATE - BUILD DATE TABLE
# ============================================================

df_dim_date = (
    spark.sql(
        f"""
        SELECT explode(
            sequence(
                to_date('{project_start_date}'),
                to_date('{project_end_date}'),
                interval 1 day
            )
        ) AS date
        """
    )
)

df_dim_date = (
    df_dim_date

    .withColumn(
        "date_key",
        F.date_format(
            "date",
            "yyyyMMdd"
        ).cast("int")
    )

    .withColumn(
        "year",
        F.year("date")
    )

    .withColumn(
        "quarter",
        F.quarter("date")
    )

    .withColumn(
        "quarter_name",
        F.concat(
            F.lit("Q"),
            F.quarter("date")
        )
    )

    .withColumn(
        "month",
        F.month("date")
    )

    .withColumn(
        "month_name",
        F.date_format(
            "date",
            "MMMM"
        )
    )

    .withColumn(
        "month_short_name",
        F.date_format(
            "date",
            "MMM"
        )
    )

    .withColumn(
        "week_of_year",
        F.weekofyear("date")
    )

    .withColumn(
        "day_of_month",
        F.dayofmonth("date")
    )

    .withColumn(
        "day_of_week",
        F.dayofweek("date")
    )

    .withColumn(
        "day_name",
        F.date_format(
            "date",
            "EEEE"
        )
    )

    .withColumn(
        "is_weekend",
        F.when(
            F.dayofweek("date").isin(1, 7),
            True
        ).otherwise(False)
    )
)

display(df_dim_date)

date,date_key,year,quarter,quarter_name,month,month_name,month_short_name,week_of_year,day_of_month,day_of_week,day_name,is_weekend
2026-01-05,20260105,2026,1,Q1,1,January,Jan,2,5,2,Monday,false
2026-01-06,20260106,2026,1,Q1,1,January,Jan,2,6,3,Tuesday,false
2026-01-07,20260107,2026,1,Q1,1,January,Jan,2,7,4,Wednesday,false
2026-01-08,20260108,2026,1,Q1,1,January,Jan,2,8,5,Thursday,false
2026-01-09,20260109,2026,1,Q1,1,January,Jan,2,9,6,Friday,false
2026-01-10,20260110,2026,1,Q1,1,January,Jan,2,10,7,Saturday,true
2026-01-11,20260111,2026,1,Q1,1,January,Jan,2,11,1,Sunday,true
2026-01-12,20260112,2026,1,Q1,1,January,Jan,3,12,2,Monday,false
2026-01-13,20260113,2026,1,Q1,1,January,Jan,3,13,3,Tuesday,false
2026-01-14,20260114,2026,1,Q1,1,January,Jan,3,14,4,Wednesday,false


In [0]:
# ============================================================
# DIM DATE - VALIDATION
# ============================================================

print(
    "Total date records:",
    df_dim_date.count()
)

print(
    "NULL date records:"
)

df_dim_date.filter(
    F.col("date").isNull()
).show()

print(
    "Duplicate dates:"
)

df_dim_date \
    .groupBy("date") \
    .count() \
    .filter(
        F.col("count") > 1
    ) \
    .show()

Total date records: 177
NULL date records:
+----+--------+----+-------+------------+-----+----------+----------------+------------+------------+-----------+--------+----------+
|date|date_key|year|quarter|quarter_name|month|month_name|month_short_name|week_of_year|day_of_month|day_of_week|day_name|is_weekend|
+----+--------+----+-------+------------+-----+----------+----------------+------------+------------+-----------+--------+----------+
+----+--------+----+-------+------------+-----+----------+----------------+------------+------------+-----------+--------+----------+

Duplicate dates:
+----+-----+
|date|count|
+----+-----+
+----+-----+



In [0]:
# ============================================================
# DIM DATE - WRITE TO GOLD
# ============================================================

df_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        "showroom_analytics.gold.dim_date"
    )

print(
    "SUCCESS: dim_date created in Gold."
)

SUCCESS: dim_date created in Gold.


In [0]:
# ============================================================
# DIM DATE - VERIFY
# ============================================================

df_gold_dim_date = spark.table(
    "showroom_analytics.gold.dim_date"
)

display(
    df_gold_dim_date
    .orderBy("date")
)

print(
    "Gold dim_date records:",
    df_gold_dim_date.count()
)

df_gold_dim_date.printSchema()

date,date_key,year,quarter,quarter_name,month,month_name,month_short_name,week_of_year,day_of_month,day_of_week,day_name,is_weekend
2026-01-05,20260105,2026,1,Q1,1,January,Jan,2,5,2,Monday,false
2026-01-06,20260106,2026,1,Q1,1,January,Jan,2,6,3,Tuesday,false
2026-01-07,20260107,2026,1,Q1,1,January,Jan,2,7,4,Wednesday,false
2026-01-08,20260108,2026,1,Q1,1,January,Jan,2,8,5,Thursday,false
2026-01-09,20260109,2026,1,Q1,1,January,Jan,2,9,6,Friday,false
2026-01-10,20260110,2026,1,Q1,1,January,Jan,2,10,7,Saturday,true
2026-01-11,20260111,2026,1,Q1,1,January,Jan,2,11,1,Sunday,true
2026-01-12,20260112,2026,1,Q1,1,January,Jan,3,12,2,Monday,false
2026-01-13,20260113,2026,1,Q1,1,January,Jan,3,13,3,Tuesday,false
2026-01-14,20260114,2026,1,Q1,1,January,Jan,3,14,4,Wednesday,false


Gold dim_date records: 177
root
 |-- date: date (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- quarter_name: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- month_short_name: string (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- is_weekend: boolean (nullable = true)



In [0]:
# ============================================================
# GOLD - ADLS EXTERNAL LOCATION
# ============================================================

gold_path = "abfss://gold@stshowroomanalytics01.dfs.core.windows.net/"

print("Gold ADLS path configured.")

Gold ADLS path configured.


In [0]:
# ============================================================
# WRITE FACT SALES TO ADLS GOLD
# ============================================================

df_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(
        gold_path + "fact_sales"
    )

print("fact_sales written successfully.")

fact_sales written successfully.


In [0]:
# ============================================================
# WRITE FACT INVENTORY TO ADLS GOLD
# ============================================================

df_fact_inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(
        gold_path + "fact_inventory"
    )

print("fact_inventory written successfully.")

fact_inventory written successfully.


In [0]:
# ============================================================
# WRITE FACT EXPENSES TO ADLS GOLD
# ============================================================

df_fact_expenses.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(
        gold_path + "fact_expenses"
    )

print("fact_expenses written successfully.")

fact_expenses written successfully.


In [0]:
# ============================================================
# WRITE FACT LEADS TO ADLS GOLD
# ============================================================

df_fact_leads.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(
        gold_path.strip() + "fact_leads"
    )

print("fact_leads written successfully.")

fact_leads written successfully.


In [0]:
# ============================================================
# WRITE DIMENSION TABLES TO ADLS GOLD
# ============================================================

gold_tables = {
    "dim_vehicle": df_dim_vehicle,
    "dim_vehicle_model": df_dim_vehicle_model,
    "dim_showroom": df_dim_showroom,
    "dim_salesperson": df_dim_salesperson,
    "dim_customer": df_dim_customer,
    "dim_campaign": df_dim_campaign,
    "dim_date": df_dim_date
}

for table_name, df in gold_tables.items():

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(
            gold_path + table_name
        )

    print(
        f"{table_name} written successfully."
    )

dim_vehicle written successfully.
dim_vehicle_model written successfully.
dim_showroom written successfully.
dim_salesperson written successfully.
dim_customer written successfully.
dim_campaign written successfully.
dim_date written successfully.


In [0]:
# ============================================================
# VERIFY GOLD DATA IN ADLS
# ============================================================

gold_table_paths = [
    "fact_sales",
    "fact_inventory",
    "fact_expenses",
    "fact_leads",
    "dim_vehicle",
    "dim_vehicle_model",
    "dim_showroom",
    "dim_salesperson",
    "dim_customer",
    "dim_campaign",
    "dim_date"
]

for table_name in gold_table_paths:

    df_check = spark.read.format("delta").load(
        gold_path + table_name
    )

    print(
        f"{table_name}: {df_check.count()} records"
    )

fact_sales: 11 records
fact_inventory: 20 records
fact_expenses: 30 records
fact_leads: 20 records
dim_vehicle: 38 records
dim_vehicle_model: 28 records
dim_showroom: 15 records
dim_salesperson: 15 records
dim_customer: 20 records
dim_campaign: 10 records
dim_date: 177 records


In [0]:
# ============================================================
# FINAL GOLD DATA QUALITY CHECK
# ============================================================

gold_validation = {
    "fact_sales": "sale_id",
    "fact_inventory": "inventory_id",
    "fact_expenses": "expense_id",
    "fact_leads": "lead_id",
    "dim_vehicle": "vehicle_id",
    "dim_vehicle_model": "model_id",
    "dim_showroom": "showroom_id",
    "dim_salesperson": "salesperson_id",
    "dim_customer": "customer_id",
    "dim_campaign": "campaign_id",
    "dim_date": "date"
}

for table_name, key_column in gold_validation.items():

    df = spark.read.format("delta").load(
        gold_path + table_name
    )

    total_records = df.count()

    null_keys = df.filter(
        F.col(key_column).isNull()
    ).count()

    duplicate_keys = (
        df.groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{table_name} | "
        f"Records: {total_records} | "
        f"NULL keys: {null_keys} | "
        f"Duplicate keys: {duplicate_keys}"
    )

fact_sales | Records: 11 | NULL keys: 0 | Duplicate keys: 0
fact_inventory | Records: 20 | NULL keys: 0 | Duplicate keys: 0
fact_expenses | Records: 30 | NULL keys: 0 | Duplicate keys: 15
fact_leads | Records: 20 | NULL keys: 0 | Duplicate keys: 0
dim_vehicle | Records: 38 | NULL keys: 0 | Duplicate keys: 0
dim_vehicle_model | Records: 28 | NULL keys: 0 | Duplicate keys: 0
dim_showroom | Records: 15 | NULL keys: 0 | Duplicate keys: 0
dim_salesperson | Records: 15 | NULL keys: 0 | Duplicate keys: 0
dim_customer | Records: 20 | NULL keys: 0 | Duplicate keys: 0
dim_campaign | Records: 10 | NULL keys: 0 | Duplicate keys: 0
dim_date | Records: 177 | NULL keys: 0 | Duplicate keys: 0


In [0]:
# ============================================================
# FACT EXPENSES - FIX DUPLICATE BUSINESS KEYS
# ============================================================

# Showroom expenses
df_showroom_expenses = (
    df_expenses
    .select(
        F.col("expense_id").cast("long").alias("source_expense_id"),
        F.col("showroom_id"),
        F.lit(None).cast("long").alias("campaign_id"),
        F.col("expense_date"),
        F.col("expense_type"),
        F.col("amount"),
        F.col("description")
    )
    .withColumn(
        "expense_category",
        F.lit("Showroom")
    )
    .withColumn(
        "expense_id",
        F.concat(
            F.lit("SHOWROOM_"),
            F.col("source_expense_id").cast("string")
        )
    )
)


# Campaign expenses
df_campaign_expenses = (
    df_campaign_expense
    .select(
        F.col("campaign_expense_id").cast("long").alias("source_expense_id"),
        F.col("showroom_id"),
        F.col("campaign_id"),
        F.col("expense_date"),
        F.lit("Campaign").alias("expense_type"),
        F.col("amount"),
        F.lit("Campaign expense").alias("description")
    )
    .withColumn(
        "expense_category",
        F.lit("Campaign")
    )
    .withColumn(
        "expense_id",
        F.concat(
            F.lit("CAMPAIGN_"),
            F.col("source_expense_id").cast("string")
        )
    )
)


# Combine both sources
df_fact_expenses = (
    df_showroom_expenses
    .unionByName(df_campaign_expenses)
)

display(df_fact_expenses)

source_expense_id,showroom_id,campaign_id,expense_date,expense_type,amount,description,expense_category,expense_id
1,1,null,2026-01-31,Rent,180000.00,Monthly showroom rent,Showroom,SHOWROOM_1
2,1,null,2026-01-31,Electricity,32000.00,Monthly electricity bill,Showroom,SHOWROOM_2
3,2,null,2026-01-31,Marketing,75000.00,Digital marketing expense,Showroom,SHOWROOM_3
4,2,null,2026-02-28,Maintenance,28000.00,Showroom maintenance,Showroom,SHOWROOM_4
5,3,null,2026-02-28,Rent,150000.00,Monthly showroom rent,Showroom,SHOWROOM_5
6,3,null,2026-02-28,Salary,220000.00,Sales staff salary,Showroom,SHOWROOM_6
7,4,null,2026-03-31,Electricity,35000.00,Monthly electricity bill,Showroom,SHOWROOM_7
8,4,null,2026-03-31,Marketing,60000.00,Local marketing campaign,Showroom,SHOWROOM_8
9,5,null,2026-03-31,Rent,175000.00,Monthly showroom rent,Showroom,SHOWROOM_9
10,5,null,2026-04-30,Maintenance,24000.00,Vehicle display area maintenance,Showroom,SHOWROOM_10


In [0]:
# ============================================================
# FACT EXPENSES - DATE ATTRIBUTES
# ============================================================

df_fact_expenses = (
    df_fact_expenses

    .withColumn(
        "expense_year",
        F.year("expense_date")
    )

    .withColumn(
        "expense_month",
        F.month("expense_date")
    )

    .withColumn(
        "expense_month_name",
        F.date_format(
            "expense_date",
            "MMMM"
        )
    )

    .withColumn(
        "expense_quarter",
        F.quarter("expense_date")
    )
)

display(df_fact_expenses)

source_expense_id,showroom_id,campaign_id,expense_date,expense_type,amount,description,expense_category,expense_id,expense_year,expense_month,expense_month_name,expense_quarter
1,1,null,2026-01-31,Rent,180000.00,Monthly showroom rent,Showroom,SHOWROOM_1,2026,1,January,1
2,1,null,2026-01-31,Electricity,32000.00,Monthly electricity bill,Showroom,SHOWROOM_2,2026,1,January,1
3,2,null,2026-01-31,Marketing,75000.00,Digital marketing expense,Showroom,SHOWROOM_3,2026,1,January,1
4,2,null,2026-02-28,Maintenance,28000.00,Showroom maintenance,Showroom,SHOWROOM_4,2026,2,February,1
5,3,null,2026-02-28,Rent,150000.00,Monthly showroom rent,Showroom,SHOWROOM_5,2026,2,February,1
6,3,null,2026-02-28,Salary,220000.00,Sales staff salary,Showroom,SHOWROOM_6,2026,2,February,1
7,4,null,2026-03-31,Electricity,35000.00,Monthly electricity bill,Showroom,SHOWROOM_7,2026,3,March,1
8,4,null,2026-03-31,Marketing,60000.00,Local marketing campaign,Showroom,SHOWROOM_8,2026,3,March,1
9,5,null,2026-03-31,Rent,175000.00,Monthly showroom rent,Showroom,SHOWROOM_9,2026,3,March,1
10,5,null,2026-04-30,Maintenance,24000.00,Vehicle display area maintenance,Showroom,SHOWROOM_10,2026,4,April,2


In [0]:
# ============================================================
# FACT EXPENSES - VALIDATE FIX
# ============================================================

total_records = df_fact_expenses.count()

null_keys = df_fact_expenses.filter(
    F.col("expense_id").isNull()
).count()

duplicate_keys = (
    df_fact_expenses
    .groupBy("expense_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total records:", total_records)
print("NULL expense IDs:", null_keys)
print("Duplicate expense IDs:", duplicate_keys)

Total records: 30
NULL expense IDs: 0
Duplicate expense IDs: 0


In [0]:
# ============================================================
# FACT EXPENSES - REWRITE ADLS GOLD
# ============================================================

df_fact_expenses.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .save(
        gold_path + "fact_expenses"
    )

print(
    "SUCCESS: fact_expenses updated in ADLS Gold."
)

SUCCESS: fact_expenses updated in ADLS Gold.


In [0]:
# ============================================================
# FACT EXPENSES - FINAL ADLS VERIFICATION
# ============================================================

df_check_expenses = (
    spark.read
    .format("delta")
    .load(
        gold_path + "fact_expenses"
    )
)

print(
    "Records:",
    df_check_expenses.count()
)

print(
    "Distinct expense IDs:",
    df_check_expenses
    .select("expense_id")
    .distinct()
    .count()
)

print("Duplicate expense IDs:")

df_check_expenses \
    .groupBy("expense_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

Records: 30
Distinct expense IDs: 30
Duplicate expense IDs:
+----------+-----+
|expense_id|count|
+----------+-----+
+----------+-----+



In [0]:
# ============================================================
# EXPORT GOLD TABLES AS PARQUET FOR SYNAPSE
# ============================================================

synapse_path = gold_path + "synapse/"

gold_tables_for_synapse = {
    "fact_sales": df_fact_sales,
    "fact_inventory": df_fact_inventory,
    "fact_expenses": df_fact_expenses,
    "fact_leads": df_fact_leads,
    "dim_vehicle": df_dim_vehicle,
    "dim_vehicle_model": df_dim_vehicle_model,
    "dim_showroom": df_dim_showroom,
    "dim_salesperson": df_dim_salesperson,
    "dim_customer": df_dim_customer,
    "dim_campaign": df_dim_campaign,
    "dim_date": df_dim_date
}

for table_name, df in gold_tables_for_synapse.items():

    df.write \
        .format("parquet") \
        .mode("overwrite") \
        .save(
            synapse_path + table_name
        )

    print(
        f"{table_name} exported as Parquet."
    )

fact_sales exported as Parquet.
fact_inventory exported as Parquet.
fact_expenses exported as Parquet.
fact_leads exported as Parquet.
dim_vehicle exported as Parquet.
dim_vehicle_model exported as Parquet.
dim_showroom exported as Parquet.
dim_salesperson exported as Parquet.
dim_customer exported as Parquet.
dim_campaign exported as Parquet.
dim_date exported as Parquet.


In [0]:
# ============================================================
# VERIFY SYNAPSE PARQUET DATA
# ============================================================

for table_name in gold_tables_for_synapse.keys():

    df_check = spark.read \
        .format("parquet") \
        .load(
            synapse_path + table_name
        )

    print(
        f"{table_name}: {df_check.count()} records"
    )

fact_sales: 11 records
fact_inventory: 20 records
fact_expenses: 30 records
fact_leads: 20 records
dim_vehicle: 38 records
dim_vehicle_model: 28 records
dim_showroom: 15 records
dim_salesperson: 15 records
dim_customer: 20 records
dim_campaign: 10 records
dim_date: 177 records
